In [ ]:
questions = {
        'contains_user_preferences': {
            'type': 'noul',
            'instructions': (
                "The state is a message written by the user. Does the user state a preference "
                "of their own: something they like, dislike, want, or want to avoid? This includes "
                "preferences about how they want to be answered (length, tone, format, language)."
            ),
            'criteria': {
                'true': (
                    "The user expresses a like, dislike, want, or wish of their own, "
                    "e.g. 'I hate emojis', 'please always answer in Dutch', 'I prefer tea over coffee'."
                ),
                'false': (
                    "The user expresses no preference of their own: the message is a question, "
                    "a one-off task, a greeting, or a statement of fact, e.g. 'what time is it?', "
                    "'summarize this file', 'hello', 'my brother likes jazz'."
                ),
            },
        },
    }

In [57]:
respond_questions = {
        'should_respond': {
            'type': 'noul',
            'instructions': (
                "The state describes a moment from the life of a user wearing PAL, a voice assistant "
                "with a camera and microphone. 'speech' is what was just heard, 'speaker' is who said it, "
                "'recent' is the conversation before it, and 'scene' describes what the camera sees. "
                "Should PAL speak up now? PAL should respond when the user is talking to PAL, or when "
                "PAL has something useful to add without being asked. PAL must stay silent when the "
                "speech is meant for someone else or is just the user thinking aloud."
            ),
            'criteria': {
                'true': (
                    "The speech is directed at PAL (it names PAL, asks the assistant something, or "
                    "continues a conversation with PAL), or there is a clear, timely reason to speak "
                    "up unprompted, such as a hazard, something the user asked PAL to watch for, or "
                    "a question the user is struggling with that PAL can answer."
                ),
                'false': (
                    "The speech is meant for another person present or on the phone, is small talk "
                    "or chatter, is the user muttering to themselves, or comes from someone other than "
                    "the user, and there is no pressing reason for PAL to interrupt."
                ),
            },
        },
    }

In [64]:
from pal.llm import get_client

big = get_client('typesafe:tev1:4b')
small = get_client('typesafe:tev1:0.8b')

In [65]:
import time

def S(speech, scene, speaker='user', recent=()):
    return {'speaker': speaker, 'speech': speech, 'recent': list(recent), 'scene': scene}

# (addressed_to_pal, worth_interrupting, state). The combined label is `addressed or interrupting`.
CASES = [
    # talking to PAL
    (True,  False, S('hey PAL, what time is it?', 'office desk, user alone')),
    (True,  False, S('and the one after that?', 'office desk, user alone',
                     recent=['user: PAL, when is my next meeting?', 'PAL: Standup at 10:00.'])),
    (True,  False, S('PAL, log that the back gate was unlocked', 'user on patrol at the back gate')),
    (True,  False, S('remind me to call my mom at five', 'kitchen, user alone')),
    (True,  False, S("what was the new guard's name again?", 'guard booth, user alone',
                     recent=['user: PAL, remember that the new guard is called Ahmed.'])),
    (True,  False, S('PAL? you there?', 'parking garage, user alone')),
    (True,  False, S("hold on, let me check... PAL, what's my schedule on Thursday?", 'user on a phone call')),
    # PAL should speak up unasked
    (False, True,  S('', 'a person in a hoodie is opening the server room door', speaker='none',
                     recent=['user: PAL, warn me if anyone enters the server room.'])),
    (False, True,  S('ugh, where did I leave my keys', 'hallway; a set of keys is on the side table')),  # debatable
    (False, True,  S('', 'smoke is coming from under the storage room door', speaker='none')),
    (False, True,  S('', 'a forklift is reversing toward the user, who is facing away', speaker='none')),
    # chatter not meant for PAL
    (False, False, S('yeah, see you tomorrow Mark!', 'office exit, a colleague waving')),
    (False, False, S('did you see the game last night?', 'coffee machine, two people chatting', speaker='colleague')),
    (False, False, S('okay, two sugars please', 'cafe counter, a barista')),
    (False, False, S('hmm, left or right... left', 'street corner, user alone')),
    (False, False, S('yeah PAL is honestly great, you should get one', 'user on a phone call')),
    (False, False, S('PAL, open the door', 'a visitor at reception speaking toward the user', speaker='visitor')),
    (False, False, S('breaking news: heavy rain expected tonight', 'break room, TV on', speaker='tv')),
    (False, False, S('can you hand me that radio?', 'a colleague standing next to the user')),
    (False, False, S('what time does our shift end?', 'a colleague standing next to the user',
                     recent=["colleague: man, I'm so tired"])),
    (False, False, S('all clear on level two, over', 'user speaking into a handheld radio')),
    (False, False, S('', 'quiet empty corridor, nothing unusual', speaker='none')),
    (False, False, S('good morning!', 'reception, receptionist smiling')),
]


def evaluate(cases, score, threshold=0.5):
    """Run `score(state) -> probability` over (expected, state) pairs and print a scorecard."""
    hits, times = 0, []
    for expected, state in cases:
        start = time.perf_counter()
        p = score(state)
        times.append(time.perf_counter() - start)
        ok = (p >= threshold) == expected
        hits += ok
        print(f"{'ok ' if ok else 'BAD'} p={p:.2f} expected={expected!s:5}  {state['speech'] or state['scene']!r}")
    print(f"\n{hits}/{len(cases)} correct, mean latency {1000 * sum(times) / len(times):.0f} ms")

In [66]:
evaluate(
    [(addressed or interrupting, state) for addressed, interrupting, state in CASES],
    lambda state: big.decide(state, respond_questions).answers['should_respond'].noul,
)

ok  p=0.99 expected=True   'hey PAL, what time is it?'
ok  p=0.65 expected=True   'and the one after that?'
ok  p=0.84 expected=True   'PAL, log that the back gate was unlocked'
ok  p=0.56 expected=True   'remind me to call my mom at five'
ok  p=0.87 expected=True   "what was the new guard's name again?"
ok  p=0.94 expected=True   'PAL? you there?'
ok  p=0.98 expected=True   "hold on, let me check... PAL, what's my schedule on Thursday?"
ok  p=0.97 expected=True   'a person in a hoodie is opening the server room door'
BAD p=0.41 expected=True   'ugh, where did I leave my keys'
ok  p=0.73 expected=True   'smoke is coming from under the storage room door'
ok  p=0.87 expected=True   'a forklift is reversing toward the user, who is facing away'
ok  p=0.00 expected=False  'yeah, see you tomorrow Mark!'
ok  p=0.01 expected=False  'did you see the game last night?'
ok  p=0.03 expected=False  'okay, two sugars please'
ok  p=0.10 expected=False  'hmm, left or right... left'
ok  p=0.09 expected=

In [67]:
evaluate(
    [(addressed or interrupting, state) for addressed, interrupting, state in CASES],
    lambda state: small.decide(state, respond_questions).answers['should_respond'].noul,
)

ok  p=0.97 expected=True   'hey PAL, what time is it?'
ok  p=0.68 expected=True   'and the one after that?'
ok  p=0.68 expected=True   'PAL, log that the back gate was unlocked'
BAD p=0.44 expected=True   'remind me to call my mom at five'
ok  p=0.68 expected=True   "what was the new guard's name again?"
ok  p=0.82 expected=True   'PAL? you there?'
ok  p=0.91 expected=True   "hold on, let me check... PAL, what's my schedule on Thursday?"
BAD p=0.22 expected=True   'a person in a hoodie is opening the server room door'
BAD p=0.38 expected=True   'ugh, where did I leave my keys'
BAD p=0.10 expected=True   'smoke is coming from under the storage room door'
BAD p=0.08 expected=True   'a forklift is reversing toward the user, who is facing away'
ok  p=0.25 expected=False  'yeah, see you tomorrow Mark!'
ok  p=0.44 expected=False  'did you see the game last night?'
BAD p=0.56 expected=False  'okay, two sugars please'
ok  p=0.35 expected=False  'hmm, left or right... left'
BAD p=0.95 expected=

## Split: `addressed_to_pal` + `worth_interrupting`

Two questions in one call. Replying when spoken to is cheap to get wrong; interrupting unasked is not, so it gets a stricter threshold.

In [60]:
STATE_DESCRIPTION = (
    "The state describes a moment from the life of a user wearing PAL, a voice assistant "
    "with a camera and microphone. 'speech' is what was just heard, 'speaker' is who said it, "
    "'recent' is the conversation before it, and 'scene' describes what the camera sees. "
)

split_questions = {
        'addressed_to_pal': {
            'type': 'noul',
            'instructions': STATE_DESCRIPTION + (
                "Is the user talking to PAL right now? Only the user's speech counts, and only "
                "when it is meant for the assistant, not for a person nearby or on the phone."
            ),
            'criteria': {
                'true': (
                    "The user is speaking to PAL: they call PAL by name to ask or tell it something, "
                    "continue a conversation PAL is part of, or give an assistant-style request "
                    "while nobody else is around to hear it."
                ),
                'false': (
                    "The speech is meant for another person, is someone other than the user "
                    "talking, merely mentions PAL, is the user thinking aloud, or there is no speech."
                ),
            },
        },
        'worth_interrupting': {
            'type': 'noul',
            'instructions': STATE_DESCRIPTION + (
                "Ignore whether anyone is talking to PAL. Is there something PAL should tell the "
                "user right now, without being asked, because waiting would be worse?"
            ),
            'criteria': {
                'true': (
                    "There is a clear, timely reason to speak up: a danger to the user, an event the "
                    "user asked PAL to watch for, or something PAL can see that solves a problem the "
                    "user is visibly stuck on."
                ),
                'false': (
                    "Nothing in the scene or conversation needs the user's attention right now; "
                    "speaking up would only interrupt."
                ),
            },
        },
    }

In [68]:
def respond_scores(state, model) -> dict[str, float]:
    """One model call; the probability for each split question."""
    answers = model.decide(state, split_questions).answers
    return {name: answers[name].noul for name in split_questions}


def respond_decision(scores, addressed_at=0.5, interrupt_at=0.8) -> str | None:
    """'reply' when spoken to, 'interrupt' when worth speaking up unasked, None to stay silent."""
    if scores['addressed_to_pal'] >= addressed_at:
        return 'reply'
    if scores['worth_interrupting'] >= interrupt_at:
        return 'interrupt'
    return None


def should_respond(state, model, **thresholds) -> str | None:
    return respond_decision(respond_scores(state, model), **thresholds)

In [69]:
# Cases that only make sense split: both signals at once, or a hazard during a conversation.
SPLIT_CASES = CASES + [
    (True,  True,  S("PAL, which way is the nearest exit?", 'smoke is filling the corridor behind the user')),
    (False, True,  S('haha yeah, totally', 'an unattended bag has sat next to the bench for ten minutes; user chatting with a colleague',
                     recent=['user: PAL, flag any unattended bags.'])),
    (False, True,  S('anyway, as I was saying', 'user talking to a visitor; a car is rolling toward the user with nobody in it')),
    (False, False, S("let's ask PAL about that later", 'a colleague standing next to the user')),
    (True,  False, S('thanks PAL, that was it', 'office desk, user alone',
                     recent=['PAL: Your badge is in your left jacket pocket.'])),  # addressed; the reply may be silence
    (True,  False, S('PAL, stop. stop talking.', 'office desk, user alone',
                     recent=['PAL: Here are the five most recent incidents. The first one'])),
    (False, False, S('', 'a colleague walks past and nods at the user', speaker='none',
                     recent=['user: PAL, warn me if anyone enters the server room.'])),  # watch rule, but not triggered
]


def expected_decision(addressed, interrupting):
    return 'reply' if addressed else 'interrupt' if interrupting else None


def evaluate_split(cases, model, **thresholds):
    counts = {'addressed_to_pal': 0, 'worth_interrupting': 0, 'decision': 0}
    times = []
    for addressed, interrupting, state in cases:
        start = time.perf_counter()
        scores = respond_scores(state, model)
        times.append(time.perf_counter() - start)

        decision, expected = respond_decision(scores, **thresholds), expected_decision(addressed, interrupting)
        a_ok = (scores['addressed_to_pal'] >= 0.5) == addressed
        i_ok = (scores['worth_interrupting'] >= 0.5) == interrupting
        counts['addressed_to_pal'] += a_ok
        counts['worth_interrupting'] += i_ok
        counts['decision'] += decision == expected

        print(f"{'ok ' if decision == expected else 'BAD'} "
              f"addr={scores['addressed_to_pal']:.2f}{' ' if a_ok else '!'} "
              f"intr={scores['worth_interrupting']:.2f}{' ' if i_ok else '!'} "
              f"{str(decision):9} expected={str(expected):9}  {state['speech'] or state['scene']!r}")

    n = len(cases)
    print('\n' + ', '.join(f'{k} {v}/{n}' for k, v in counts.items())
          + f', mean latency {1000 * sum(times) / n:.0f} ms')

In [71]:


evaluate_split(SPLIT_CASES, big)

ok  addr=0.95  intr=0.20  reply     expected=reply      'hey PAL, what time is it?'
ok  addr=0.80  intr=0.22  reply     expected=reply      'and the one after that?'
ok  addr=0.98  intr=0.50! reply     expected=reply      'PAL, log that the back gate was unlocked'
ok  addr=0.91  intr=0.08  reply     expected=reply      'remind me to call my mom at five'
ok  addr=0.85  intr=0.82! reply     expected=reply      "what was the new guard's name again?"
ok  addr=0.91  intr=0.10  reply     expected=reply      'PAL? you there?'
ok  addr=0.93  intr=0.20  reply     expected=reply      "hold on, let me check... PAL, what's my schedule on Thursday?"
ok  addr=0.27  intr=0.94  interrupt expected=interrupt  'a person in a hoodie is opening the server room door'
ok  addr=0.38  intr=0.95  interrupt expected=interrupt  'ugh, where did I leave my keys'
BAD addr=0.03  intr=0.62  None      expected=interrupt  'smoke is coming from under the storage room door'
ok  addr=0.02  intr=0.92  interrupt expected=int

In [70]:


evaluate_split(SPLIT_CASES, small)

ok  addr=0.99  intr=0.35  reply     expected=reply      'hey PAL, what time is it?'
ok  addr=0.92  intr=0.38  reply     expected=reply      'and the one after that?'
ok  addr=0.98  intr=0.56! reply     expected=reply      'PAL, log that the back gate was unlocked'
ok  addr=0.88  intr=0.41  reply     expected=reply      'remind me to call my mom at five'
ok  addr=0.92  intr=0.38  reply     expected=reply      "what was the new guard's name again?"
ok  addr=0.98  intr=0.38  reply     expected=reply      'PAL? you there?'
ok  addr=0.98  intr=0.50! reply     expected=reply      "hold on, let me check... PAL, what's my schedule on Thursday?"
BAD addr=0.27  intr=0.56  None      expected=interrupt  'a person in a hoodie is opening the server room door'
BAD addr=0.75! intr=0.41! reply     expected=interrupt  'ugh, where did I leave my keys'
BAD addr=0.25  intr=0.29! None      expected=interrupt  'smoke is coming from under the storage room door'
BAD addr=0.18  intr=0.32! None      expected=int